In [1]:
import pandas as pd
import psycopg2

from pathlib import Path
from sqlalchemy import create_engine, text

In [2]:
hosting = "dpg-dc3v9gmi0phs738mhs1g-a.oregon-postgres.render.com"
puerto = "5432"
nombre_db = "bbdd_equipo_2"
usuario = "equipo"
pswd = "N4zZY6CQfV637bflevxo7RgtqmxMUF0d"

In [3]:
engine = create_engine(
    f"postgresql+psycopg2://{usuario}:{pswd}@{hosting}:{puerto}/{nombre_db}",
    connect_args={"sslmode": "require"}
)

In [4]:
pd.read_sql("SELECT current_database();", engine)

,current_database
0,bbdd_equipo_2


In [6]:
sql_crear_tablas = text("""
CREATE TABLE IF NOT EXISTS vertical (
    id_vertical INT PRIMARY KEY,
    vertical VARCHAR(100) NOT NULL
);

CREATE TABLE IF NOT EXISTS campus (
    id_campus INT PRIMARY KEY,
    campus VARCHAR(100) NOT NULL
);

CREATE TABLE IF NOT EXISTS modalidad (
    id_modalidad INT PRIMARY KEY,
    modalidad VARCHAR(100) NOT NULL
);

CREATE TABLE IF NOT EXISTS promocion (
    id_promocion INT PRIMARY KEY,
    id_campus INT NOT NULL REFERENCES campus(id_campus),
    id_vertical INT NOT NULL REFERENCES vertical(id_vertical),
    promocion VARCHAR(120) NOT NULL,
    fecha_comienzo DATE
);

CREATE TABLE IF NOT EXISTS profesor (
    id_profesor INT PRIMARY KEY,
    nombre VARCHAR(150) NOT NULL,
    id_modalidad INT NOT NULL REFERENCES modalidad(id_modalidad),
    id_campus INT NOT NULL REFERENCES campus(id_campus)
);

CREATE TABLE IF NOT EXISTS proyecto (
    id_proyecto INT PRIMARY KEY,
    id_vertical INT NOT NULL REFERENCES vertical(id_vertical),
    proyecto VARCHAR(150) NOT NULL
);

CREATE TABLE IF NOT EXISTS alumno (
    id_alumno INT PRIMARY KEY,
    nombre VARCHAR(150) NOT NULL,
    email VARCHAR(150),
    id_promocion INT NOT NULL REFERENCES promocion(id_promocion)
);

CREATE TABLE IF NOT EXISTS evaluacion (
    id_evaluacion INT PRIMARY KEY,
    id_proyecto INT NOT NULL REFERENCES proyecto(id_proyecto),
    id_alumno INT NOT NULL REFERENCES alumno(id_alumno),
    nota NUMERIC(5,2)
);
""")

with engine.begin() as conn:
    conn.execute(sql_crear_tablas)

print("8 tablas creadas (o ya existentes) en bbdd_equipo_2.")

8 tablas creadas (o ya existentes) en bbdd_equipo_2.


In [ ]:
csv_dir = Path(".")

tablas_csv = [
    ("vertical.csv", "vertical"),
    ("campus.csv", "campus"),
    ("modalidad.csv", "modalidad"),
    ("promocion.csv", "promocion"),
    ("profesor.csv", "profesor"),
    ("proyecto.csv", "proyecto"),
    ("alumno.csv", "alumno"),
    ("evaluacion.csv", "evaluacion"),
]



for archivo, tabla in tablas_csv:
    df_temp = pd.read_csv(csv_dir / archivo)
    df_temp.columns = df_temp.columns.str.strip().str.lower()
    df_temp.to_sql(tabla, con=engine, if_exists="append", index=False)
    print(f"Cargada '{tabla}' ({len(df_temp)} filas desde {archivo}).")

Cargada 'vertical' (2 filas desde vertical.csv).
Cargada 'campus' (2 filas desde campus.csv).
Cargada 'modalidad' (2 filas desde modalidad.csv).
Cargada 'promocion' (4 filas desde promocion.csv).
Cargada 'profesor' (10 filas desde profesor.csv).
Cargada 'proyecto' (10 filas desde proyecto.csv).
Cargada 'alumno' (52 filas desde alumno.csv).


DatabaseError: Execution failed on sql 'INSERT INTO evaluacion (id_evaluacion, id_proyecto, id_alumno, nota) VALUES (:id_evaluacion, :id_proyecto, :id_alumno, :nota)': (psycopg2.errors.InvalidTextRepresentation) invalid input syntax for type numeric: "Apto"
LINE 1: ...n, id_proyecto, id_alumno, nota) VALUES (1, 1, 1, 'Apto'), (...
                                                             ^

[SQL: INSERT INTO evaluacion (id_evaluacion, id_proyecto, id_alumno, nota) VALUES (%(id_evaluacion__0)s, %(id_proyecto__0)s, %(id_alumno__0)s, %(nota__0)s), (%(id_evaluacion__1)s, %(id_proyecto__1)s, %(id_alumno__1)s, %(nota__1)s), (%(id_evaluacion__2)s, % ... 20864 characters truncated ... s, %(nota__258)s), (%(id_evaluacion__259)s, %(id_proyecto__259)s, %(id_alumno__259)s, %(nota__259)s)]
[parameters: {'id_proyecto__0': 1, 'id_evaluacion__0': 1, 'id_alumno__0': 1, 'nota__0': 'Apto', 'id_proyecto__1': 1, 'id_evaluacion__1': 2, 'id_alumno__1': 2, 'nota__1': 'Apto', 'id_proyecto__2': 1, 'id_evalua

In [9]:
with engine.begin() as conn:
    conn.execute(text("ALTER TABLE evaluacion ALTER COLUMN nota TYPE VARCHAR(50);"))

df_eval = pd.read_csv(csv_dir / "evaluacion.csv")
df_eval.columns = df_eval.columns.str.strip().str.lower()
df_eval.to_sql("evaluacion", con=engine, if_exists="append", index=False)
print(f"Cargada 'evaluacion' ({len(df_eval)} filas desde evaluacion.csv).")

Cargada 'evaluacion' (260 filas desde evaluacion.csv).


In [10]:
sql_conteo = """
SELECT 'vertical' AS tabla, COUNT(*) AS filas FROM vertical
UNION ALL SELECT 'campus', COUNT(*) FROM campus
UNION ALL SELECT 'modalidad', COUNT(*) FROM modalidad
UNION ALL SELECT 'promocion', COUNT(*) FROM promocion
UNION ALL SELECT 'profesor', COUNT(*) FROM profesor
UNION ALL SELECT 'proyecto', COUNT(*) FROM proyecto
UNION ALL SELECT 'alumno', COUNT(*) FROM alumno
UNION ALL SELECT 'evaluacion', COUNT(*) FROM evaluacion
ORDER BY tabla;
"""

display(pd.read_sql(sql_conteo, engine))

consulta_valencia = """
SELECT
    pr.id_profesor,
    pr.nombre,
    m.modalidad,
    c.campus
FROM profesor pr
JOIN modalidad m ON pr.id_modalidad = m.id_modalidad
JOIN campus c ON pr.id_campus = c.id_campus
WHERE c.campus = 'Valencia'
ORDER BY pr.id_profesor;
"""

display(pd.read_sql(consulta_valencia, engine))

,tabla,filas
0,alumno,52
1,campus,2
2,evaluacion,260
3,modalidad,2
4,profesor,10
5,promocion,4
6,proyecto,10
7,vertical,2


,id_profesor,nombre,modalidad,campus
0,4,Rosalva Ayuso,Presencial,Valencia
1,5,Ana Sofía Ferrer,Presencial,Valencia
2,8,Mario Prats,Online,Valencia
